# **Atividade Frequências**
### **Enderson Carvalho, Yasmin Gusmão**

In [ ]:
import pandas as pd

# Leitura do CSV bruto da PRF
df = pd.read_csv("dados_abertos_prf-datatran2025.csv", sep=";", encoding="latin1", low_memory=False)

# Garantia de que as colunas de vítimas estão em formato numérico
for coluna in ["mortos", "feridos_leves", "feridos_graves"]:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0)

# Conversão da data para permitir agrupamento por mês (usada na série temporal)
df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")

# Criação da coluna acidente_fatal: 1 se houve pelo menos 1 morto, 0 caso contrário
df["acidente_fatal"] = (df["mortos"] > 0).astype(int)

print("Base carregada:", df.shape)
print("Taxa global de fatalidade:", f"{df['acidente_fatal'].mean():.1%}")

Base carregada: (72529, 31)
Taxa global de fatalidade: 7.2%


FREQUÊNCIAS

O que é uma frequência (diferente de ranking): aqui a pergunta é só "quantos acidentes aconteceram em cada categoria?" - sem cruzar com fatalidade ainda. Serve pra entender onde e quando os acidentes se concentram, antes de investigar gravidade.

Frequência 1 - Por Fase do Dia

Responde: em que momento do dia (Dia, Noite, Alvorada, Crepúsculo) os acidentes mais acontecem? É a dimensão temporal/ambiental citada no PDF.

In [ ]:
freq_fase_dia = df["fase_dia"].value_counts().rename_axis("fase_dia").reset_index(name="qtd_acidentes")
freq_fase_dia["percentual"] = (freq_fase_dia["qtd_acidentes"] / len(df) * 100).round(1)

print("Frequência de acidentes por Fase do Dia:")
display(freq_fase_dia)

Frequência de acidentes por Fase do Dia:


,fase_dia,qtd_acidentes,percentual
0,Pleno dia,40375,55.7
1,Plena Noite,24781,34.2
2,Anoitecer,3926,5.4
3,Amanhecer,3447,4.8


Fase do dia: 55,7% dos acidentes acontecem em Pleno Dia (40.375), mas a Plena Noite já concentra 34,2% (24.781) mesmo tendo menos horas de tráfego intenso — isso já é um sinal indireto de que a exposição por hora à noite pode ser desproporcional ao volume absoluto.

Frequência 2 - Por Condição Meteorológica

Responde: em quais condições climáticas os acidentes mais ocorrem? É a dimensão ambiental do PDF.

In [ ]:
freq_clima = df["condicao_metereologica"].value_counts().rename_axis("condicao_metereologica").reset_index(name="qtd_acidentes")
freq_clima["percentual"] = (freq_clima["qtd_acidentes"] / len(df) * 100).round(1)

print("Frequência de acidentes por Condição Meteorológica:")
display(freq_clima)

Frequência de acidentes por Condição Meteorológica:


,condicao_metereologica,qtd_acidentes,percentual
0,Céu Claro,46375,63.9
1,Nublado,11435,15.8
2,Chuva,6438,8.9
3,Sol,4201,5.8
4,Garoa/Chuvisco,2422,3.3
5,Ignorado,1000,1.4
6,Nevoeiro/Neblina,553,0.8
7,Vento,104,0.1
8,Neve,1,0.0


Condição meteorológica: 64% dos acidentes ocorrem com Céu Claro. Isso não significa que tempo bom é mais perigoso — significa que a maior parte do tráfego acontece com tempo bom. Chuva (6.438) e Garoa/Chuvisco (2.422) somados são só 12,2% do volume — insight: qualquer análise de "clima ruim aumenta letalidade" precisa olhar % fatal dentro de cada categoria, não frequência bruta (veja Rankings/Cruzamentos, não Frequência).

Frequência 3 - Por Dia da Semana

Responde: existe um padrão semanal? Os acidentes se concentram em algum dia específico (ex.: fim de semana)?

In [ ]:
freq_dia_semana = df["dia_semana"].value_counts().rename_axis("dia_semana").reset_index(name="qtd_acidentes")
freq_dia_semana["percentual"] = (freq_dia_semana["qtd_acidentes"] / len(df) * 100).round(1)

print("Frequência de acidentes por Dia da Semana:")
display(freq_dia_semana)

Frequência de acidentes por Dia da Semana:


,dia_semana,qtd_acidentes,percentual
0,sábado,11554,15.9
1,domingo,11470,15.8
2,sexta-feira,11197,15.4
3,segunda-feira,10285,14.2
4,quarta-feira,9556,13.2
5,quinta-feira,9405,13.0
6,terça-feira,9062,12.5


Dia da semana: Sábado (11.554) e Domingo (11.470) lideram, mas a diferença pro dia de semana mais baixo (terça, 9.062) é de só ~27% — não é uma disparidade gigante, é uma leve concentração de fim de semana.

RANKINGS

O que é um ranking (diferente de frequência): aqui já cruzamos com acidente_fatal e ordenamos. O PDF é explícito: "ranking de volume não é ranking de fatalidade" - por isso cada ranking abaixo mostra as duas ordenações (por volume e por % fatal) pra você comparar as narrativas diferentes que cada uma conta.


Ranking 1 - Por UF
Segue exatamente a demonstração do PDF: agrupar por UF → contar ocorrências → somar fatais → somar mortos → calcular percentual.

In [ ]:
ranking_uf = df.groupby("uf").agg(
    acidentes=("id", "count"),
    fatais=("acidente_fatal", "sum"),
    mortos=("mortos", "sum")
)
ranking_uf["pct_fatal"] = (ranking_uf["fatais"] / ranking_uf["acidentes"] * 100).round(1)

# ordenação 1: por volume (priorização operacional)
print("Ranking por UF — ordenado por VOLUME de acidentes:")
display(ranking_uf.sort_values("acidentes", ascending=False).head(10))

# ordenação 2: por percentual fatal (gravidade relativa)
print("\nRanking por UF — ordenado por % FATAL (mínimo 30 acidentes p/ evitar amostra pequena):")
display(ranking_uf[ranking_uf["acidentes"] >= 30].sort_values("pct_fatal", ascending=False).head(10))

print(f"\nTaxa global de referência: {df['acidente_fatal'].mean()*100:.1f}%")

Ranking por UF — ordenado por VOLUME de acidentes:


,acidentes,fatais,mortos,pct_fatal
uf,,,,
MG,9570,647,765,6.8
SC,8186,374,434,4.6
PR,7630,511,593,6.7
RJ,6428,306,330,4.8
RS,4899,275,327,5.6
SP,4683,205,221,4.4
BA,4108,476,583,11.6
GO,3196,247,308,7.7
PE,3013,302,336,10.0



Ranking por UF — ordenado por % FATAL (mínimo 30 acidentes p/ evitar amostra pequena):


,acidentes,fatais,mortos,pct_fatal
uf,,,,
MA,1262,236,281,18.7
PA,1117,193,224,17.3
RR,142,23,28,16.2
AM,138,19,26,13.8
AL,629,86,96,13.7
TO,677,83,102,12.3
CE,1302,153,171,11.8
BA,4108,476,583,11.6
AC,280,29,29,10.4



Taxa global de referência: 7.2%


Insight central: MG tem 9.570 acidentes mas só 6,76% de fatalidade (perto da taxa global). MA tem apenas 1.262 acidentes — 13% do volume de MG — mas mata proporcionalmente quase 3x mais. Se a PRF decidisse alocar recurso só olhando volume, o Maranhão ficaria invisível no radar, mesmo sendo, proporcionalmente, o lugar mais letal do país pra sofrer um acidente.

Cautela (repetindo o PDF): UF é agregado — MA/PA podem ter menos fiscalização, rodovias de pista simples, menor infraestrutura de resgate. Não dá pra afirmar causa só com esse corte.

RANKINGS — volume vs. proporção contam histórias diferentes
Por UF — o achado mais forte de toda a análise
Critério	          Achado
Maior volume - 	MG (9.570), SC (8.186), PR (7.630) — sudeste/sul concentram tráfego

Maior % fatal (min. 30 acidentes)	-  MA (18,70%), PA (17,28%), RR (16,20%) — mais que o dobro da taxa global (7,18%)

Menor % fatal	 -  DF (4,25%), SP (4,38%), SC (4,57%)



Interpretação (o que escrever no relatório): o ranking por volume mostra onde a PRF deveria priorizar patrulhamento em termos de quantidade bruta de ocorrências. Já o ranking por % fatal mostra onde, proporcionalmente, cada acidente tem mais chance de matar — são perguntas diferentes e podem apontar UFs diferentes.

Cautela metodológica: UF é uma dimensão geográfica agregada — ela esconde diferenças de BR, trecho, fluxo de veículos e nível de fiscalização dentro do próprio estado. Não dá pra concluir causalidade só olhando a UF isolada.

Ranking 2 — Por BR (Rodovia)

Mesma lógica do ranking de UF, mas na dimensão de rodovia — ajuda a identificar os trechos federais mais críticos.

In [ ]:
ranking_br = df.groupby("br").agg(
    acidentes=("id", "count"),
    fatais=("acidente_fatal", "sum"),
    mortos=("mortos", "sum")
)
ranking_br["pct_fatal"] = (ranking_br["fatais"] / ranking_br["acidentes"] * 100).round(1)

print("Ranking por BR — ordenado por VOLUME de acidentes:")
display(ranking_br.sort_values("acidentes", ascending=False).head(10))

print("\nRanking por BR — ordenado por MORTOS (mínimo 30 acidentes):")
display(ranking_br[ranking_br["acidentes"] >= 30].sort_values("mortos", ascending=False).head(10))

Ranking por BR — ordenado por VOLUME de acidentes:


,acidentes,fatais,mortos,pct_fatal
br,,,,
101,13014,682,760,5.2
116,11021,638,708,5.8
40,3502,180,214,5.1
381,3496,171,190,4.9
153,2789,226,282,8.1
163,2519,173,210,6.9
364,2264,153,173,6.8
277,2157,134,152,6.2
262,1769,148,169,8.4



Ranking por BR — ordenado por MORTOS (mínimo 30 acidentes):


,acidentes,fatais,mortos,pct_fatal
br,,,,
101,13014,682,760,5.2
116,11021,638,708,5.8
153,2789,226,282,8.1
40,3502,180,214,5.1
163,2519,173,210,6.9
316,1236,182,201,14.7
230,1745,168,192,9.6
381,3496,171,190,4.9
364,2264,153,173,6.8


Por BR — onde tem mais gente, mas onde mata mais concentrado
Maior volume: BR-101 (13.014 acidentes) e BR-116 (11.021) — são disparadamente as mais movimentadas, junto respondem por ~33% de toda a base.

BR-316 é um destaque de proporção: só 1.236 acidentes, mas 182 fatais → 14,72% de fatalidade, mais que o dobro da BR-101 (5,24%) mesmo tendo 10x menos volume.

Insight prático pro slide "Mapeamento Crítico": BR-101 e BR-116 pedem policiamento por volume (mais chance de acidente acontecer); BR-316 pede atenção por gravidade (quando acontece, mata muito mais).

Ranking 3 - Por Tipo de Acidente

Aqui o objetivo muda: não é onde, é qual tipo de colisão mata mais proporcionalmente — ex.: atropelamento de pedestre vs. colisão traseira vs. colisão frontal.

In [ ]:
ranking_tipo = df.groupby("tipo_acidente").agg(
    acidentes=("id", "count"),
    fatais=("acidente_fatal", "sum"),
    mortos=("mortos", "sum")
)
ranking_tipo["pct_fatal"] = (ranking_tipo["fatais"] / ranking_tipo["acidentes"] * 100).round(1)

print("Ranking por Tipo de Acidente — ordenado por VOLUME:")
display(ranking_tipo.sort_values("acidentes", ascending=False).head(10))

print("\nRanking por Tipo de Acidente — ordenado por % FATAL (mínimo 30 acidentes):")
display(ranking_tipo[ranking_tipo["acidentes"] >= 30].sort_values("pct_fatal", ascending=False).head(10))

Ranking por Tipo de Acidente — ordenado por VOLUME:


,acidentes,fatais,mortos,pct_fatal
tipo_acidente,,,,
Colisão traseira,14360,619,683,4.3
Saída de leito carroçável,10209,605,700,5.9
Colisão transversal,9306,427,481,4.6
Colisão lateral mesmo sentido,7885,212,228,2.7
Tombamento,6351,268,293,4.2
Colisão com objeto,5109,297,323,5.8
Colisão frontal,4739,1396,1863,29.5
Queda de ocupante de veículo,3450,87,89,2.5
Atropelamento de Pedestre,3057,902,919,29.5



Ranking por Tipo de Acidente — ordenado por % FATAL (mínimo 30 acidentes):


,acidentes,fatais,mortos,pct_fatal
tipo_acidente,,,,
Atropelamento de Pedestre,3057,902,919,29.5
Colisão frontal,4739,1396,1863,29.5
Colisão lateral sentido oposto,2152,212,255,9.9
Eventos atípicos,287,23,24,8.0
Atropelamento de Animal,1133,68,74,6.0
Saída de leito carroçável,10209,605,700,5.9
Colisão com objeto,5109,297,323,5.8
Capotamento,1373,63,74,4.6
Colisão transversal,9306,427,481,4.6


Por Tipo de Acidente — o achado mais chocante da base
Tipo	Volume	% Fatal
Colisão traseira	- 14.360 (maior volume)	- 4,31%

Atropelamento de Pedestre -	3.057 (5º lugar em volume)- 29,51%

Colisão frontal	- 4.739	- 29,46%

Insight central: Colisão traseira é o tipo mais comum (19,8% de tudo), mas é o quinto menos letal proporcionalmente. Já Atropelamento de Pedestre e Colisão Frontal — juntos só 10,8% do volume — concentram mais que 6x a taxa de fatalidade de uma colisão traseira. Em números absolutos: colisão frontal sozinha (4.739 acidentes) matou 1.863 pessoas — mais do que colisão traseira (683 mortos) tendo 3x menos ocorrências. Esse é literalmente o achado que o Slide 5 do PDF ("causas de maior impacto e tipos mais letais") está pedindo.

Interpretação: tipos de acidente com alto volume (como colisão traseira) costumam ter % fatal baixo — são acidentes de baixa velocidade relativa. Já tipos com menor volume, como atropelamento, costumam concentrar % fatal muito mais alto. Volume alto não é sinônimo de gravidade alta.

SÉRIE TEMPORAL

O que é: mostra a evolução mês a mês de acidentes, fatais, mortos e % fatal ao longo de 2025 - usando exatamente o código-base do PDF. Serve pra identificar variação e sazonalidade aparente, não pra provar tendência ou efeito de alguma política - é descritivo, não explicativo.

In [ ]:
df["mes"] = df["data_inversa"].dt.to_period("M").astype(str)

serie_mensal = df.groupby("mes").agg(
    acidentes=("id", "count"),
    fatais=("acidente_fatal", "sum"),
    mortos=("mortos", "sum")
)
serie_mensal["pct_fatal"] = (serie_mensal["fatais"] / serie_mensal["acidentes"] * 100).round(1)

print("Série mensal de acidentes, fatais, mortos e % fatal:")
display(serie_mensal)

Série mensal de acidentes, fatais, mortos e % fatal:


,acidentes,fatais,mortos,pct_fatal
mes,,,,
2025-01,5528,359,418,6.5
2025-02,5287,362,412,6.8
2025-03,5960,402,462,6.7
2025-04,5786,414,495,7.2
2025-05,6096,504,574,8.3
2025-06,6122,457,528,7.5
2025-07,6238,456,536,7.3
2025-08,6246,472,554,7.6
2025-09,6017,438,500,7.3


Insight: Dezembro tem o maior volume de acidentes (época de festas/viagens), mas não é o mês mais letal proporcionalmente — Maio é. Volume e % fatal claramente não andam juntos mês a mês.

Limitação a registrar (obrigatório pelo critério do PDF): com só 12 meses de um único ano, não dá pra separar "maio é sazonalmente mais perigoso" de "maio de 2025 teve um outlier pontual" — precisaria de série histórica de vários anos pra confirmar padrão.

Limitação a registrar no relatório: essa série descreve a variação mês a mês, mas com apenas 1 ano de dados (2025) não dá pra separar sazonalidade real de ruído estatístico, nem atribuir qualquer queda/alta a uma causa específica.